# SHViT + DTD on Google Colab

This notebook walks through:
1. Enabling GPU and checking the environment
2. Cloning SHViT and installing dependencies
3. Downloading pretrained SHViT-S4 weights
4. Downloading DTD with Tip-Adapter style preprocessing
5. Verifying the model loads and runs inference
6. Running SHViT's official eval script

> **Before running:** Go to `Runtime → Change runtime type → T4 GPU`
> All outputs from this stage are saved under `/content/CV_Research_Paper_DTD/`.


## 0. Check GPU & environment

In [1]:
import torch

print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))
    print('VRAM            :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

import sys
print('Python version  :', sys.version.split()[0])

PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU             : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM            : 102.0 GB
Python version  : 3.12.13


## 1. (Optional) Mount Google Drive

DTD is small (~600 MB, 5,640 images across 47 texture categories), so
keeping it on Drive across sessions is optional — re-downloading every
session is quick. Mounting still saves the SHViT checkpoints, which is
handy if you want to resume fine-tuning.


In [2]:
USE_DRIVE = False   # set True to persist data + outputs in Google Drive

OUT_ROOT = '/content/CV_Research_Paper_DTD'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = '/content/drive/MyDrive/dtd_data'
    OUT_ROOT  = '/content/drive/MyDrive/CV_Research_Paper_DTD'
else:
    DATA_ROOT = '/content/dtd_data'

import os
os.makedirs(OUT_ROOT, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

print('Dataset will be stored at:', DATA_ROOT)
print('Outputs will be written under:', OUT_ROOT)


Dataset will be stored at: /content/dtd_data
Outputs will be written under: /content/CV_Research_Paper_DTD


## 2. Clone SHViT

In [3]:
import os

if not os.path.isdir('/content/SHViT'):
    !git clone https://github.com/ysj9909/SHViT.git /content/SHViT
else:
    print('SHViT already cloned, skipping.')

!ls /content/SHViT

Cloning into '/content/SHViT'...
remote: Enumerating objects: 183, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (152/152), done.
remote: Total 183 (delta 84), reused 82 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (183/183), 168.52 KiB | 1.29 MiB/s, done.
Resolving deltas: 100% (84/84), done.
acc_vs_thro.png  engine.py	  losses.py  README.md	       utils.py
data		 export_model.py  main.py    requirements.txt
downstream	 LICENSE	  model      speed_test.py


## 3. Install dependencies

Colab ships with PyTorch 2.x which satisfies SHViT's `>=1.11` requirement,
so we only need to install the extra packages from `requirements.txt`.

`--no-deps` on timm avoids overwriting Colab's torch/torchvision with
the older versions timm 0.5.4 would otherwise pull in.

In [4]:
# scikit-image==0.19.3 from SHViT's requirements has no wheels for Python
# 3.12 (Colab's default) — and we don't actually need it. Install only what
# the SHViT model architecture needs.
!pip install -q timm==0.5.4 --no-deps
!pip install -q einops==0.4.1 easydict
print('Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 7.5 MB/s eta 0:00:00
Dependencies installed.


## 4. Download SHViT-S4 pretrained weights

In [5]:
WEIGHTS_DIR = '/content/weights'
WEIGHTS_PATH = f'{WEIGHTS_DIR}/shvit_s4.pth'

os.makedirs(WEIGHTS_DIR, exist_ok=True)

if not os.path.exists(WEIGHTS_PATH):
    !wget -q --show-progress \
        https://github.com/ysj9909/SHViT/releases/download/v1.0/shvit_s4.pth \
        -O {WEIGHTS_PATH}
else:
    print('Weights already downloaded, skipping.')

size_mb = os.path.getsize(WEIGHTS_PATH) / 1e6
print(f'Checkpoint size: {size_mb:.1f} MB  ->  {WEIGHTS_PATH}')

/content/weights/sh 100%[===================>] 254.37M  35.2MB/s    in 6.5s    
Checkpoint size: 266.7 MB  ->  /content/weights/shvit_s4.pth


## 5. Download DTD with Tip-Adapter style preprocessing

This clones the DTD project and runs `prepare_dtd.py`, which
downloads the dataset via `torchvision.datasets.DTD`, re-layouts the
image tree into the Tip-Adapter folder convention
(`dtd/images/<letter>/<scene>/...`), and prepares a CoOp-style
`split_zhou_DescribableTextures.json` (using gdown when available, otherwise generating
a deterministic 50/20/30 train/val/test split).


In [6]:
import os, shutil

REPO_DIR = '/content/Vision_Project_spring_26'
if not os.path.isdir(REPO_DIR):
    !git clone -b Vision_Project_spring_26_DTD \
        https://github.com/saif-farid-tech/Vision_Project_spring_26.git {REPO_DIR}

# Make sure the prepare script + dataset helpers are importable from /content
for fname in [
    'prepare_dtd.py',
    'splits.py',
    'metrics.py',
    'augmentation.py',
]:
    shutil.copy(f'{REPO_DIR}/{fname}', f'/content/{fname}')

# Vendor the datasets/ package (Tip-Adapter utilities)
DATASETS_DST = '/content/datasets'
if os.path.isdir(DATASETS_DST):
    shutil.rmtree(DATASETS_DST)
shutil.copytree(f'{REPO_DIR}/datasets', DATASETS_DST)

!pip install -q gdown
!python /content/prepare_dtd.py --root {DATA_ROOT}


Cloning into '/content/Vision_Project_spring_26'...
remote: Enumerating objects: 916, done.
remote: Counting objects: 100% (347/347), done.
remote: Compressing objects: 100% (182/182), done.
remote: Total 916 (delta 236), reused 210 (delta 160), pack-reused 569 (from 1)
Receiving objects: 100% (916/916), 474.53 MiB | 39.09 MiB/s, done.
Resolving deltas: 100% (479/479), done.
[dtd] downloading DTD via torchvision into /content/dtd_data ...
100% 625M/625M [00:34<00:00, 18.0MB/s]
[dtd] moving /content/dtd_data/dtd/dtd/images -> /content/dtd_data/dtd/images
[dtd] trying official split download via gdown: https://drive.google.com/uc?id=1u3_QfB467jqHgNXC00UIzbLZRQCg2S7x
Downloading...
From: https://drive.google.com/uc?id=1u3_QfB467jqHgNXC00UIzbLZRQCg2S7x
To: /content/dtd_data/dtd/split_zhou_DescribableTextures.json
100% 564k/564k [00:00<00:00, 6.76MB/s]
[dtd] official split saved to /content/dtd_data/dtd/split_zhou_DescribableTextures.json
Reading split from /content/dtd_data/dtd/split_zhou_

In [7]:
# Sanity check
from pathlib import Path
ds_root = Path(DATA_ROOT) / 'dtd'
img_root = ds_root / 'images'
split_json = ds_root / 'split_zhou_DescribableTextures.json'

n_classes = sum(1 for p in img_root.iterdir() if p.is_dir()) if img_root.exists() else 0
n_img = sum(1 for _ in img_root.rglob('*.jpg')) if img_root.exists() else 0
print(f'On-disk:  {n_classes} category folders, {n_img} images under {img_root}')
print(f'Split    : {split_json} (exists: {split_json.exists()})')


On-disk:  47 category folders, 5640 images under /content/dtd_data/dtd/images
Split    : /content/dtd_data/dtd/split_zhou_DescribableTextures.json (exists: True)


## 6. Verify model loads and runs inference

Loads the SHViT-S4 checkpoint and runs 50 DTD images through it.
Predictions are ImageNet class indices (not DTD labels) — accuracy
will be ~zero until the model is fine-tuned. The goal here is just to
confirm no import / shape errors occur.


In [8]:
import sys, time, pathlib
import torch
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode

sys.path.insert(0, '/content/SHViT')

from model import shvit
import timm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

model_shvit = timm.create_model('shvit_s4', pretrained=False, num_classes=1000)

ckpt = torch.load(WEIGHTS_PATH, map_location='cpu', weights_only=False)
state_dict = ckpt.get('model', ckpt)
missing, unexpected = model_shvit.load_state_dict(state_dict, strict=False)
print(f'Missing keys: {len(missing)}   Unexpected keys: {len(unexpected)}')

model_shvit.to(DEVICE).eval()
print('Model loaded successfully.')


Using device: cuda
Missing keys: 0   Unexpected keys: 0
Model loaded successfully.


In [9]:
NUM_IMAGES = 50

# CLIP / Tip-Adapter normalization
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD  = (0.26862954, 0.26130258, 0.27577711)

transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(CLIP_MEAN, CLIP_STD),
])

# DTD has one folder per texture category under images/.
import pathlib
image_root = pathlib.Path(DATA_ROOT) / 'dtd' / 'images'
items = []
for cls_dir in sorted(image_root.iterdir()):
    if not cls_dir.is_dir():
        continue
    for img_path in sorted(cls_dir.glob('*.jpg')):
        items.append((img_path, cls_dir.name))
        if len(items) >= NUM_IMAGES:
            break
    if len(items) >= NUM_IMAGES:
        break

print(f'Running inference on {len(items)} images ...')
t0 = time.perf_counter()
results = []
with torch.no_grad():
    for img_path, true_class in items:
        x = transform(Image.open(img_path).convert('RGB')).unsqueeze(0).to(DEVICE)
        pred = int(model_shvit(x).argmax(1).item())
        results.append((img_path.name, true_class, pred))

elapsed = time.perf_counter() - t0
print(f'\n{"Image":<30} {"True class":<25} {"Pred idx":>8}')
print('-' * 65)
for name, cls, pred in results[:15]:
    print(f'{name:<30} {cls:<25} {pred:>8}')
print(f'\nTotal: {elapsed:.2f}s  ({elapsed/len(results)*1000:.1f} ms/image)')
print('\n[OK] Model ran without errors.')


Running inference on 50 images ...

Image                          True class                Pred idx
-----------------------------------------------------------------
banded_0002.jpg                banded                         794
banded_0004.jpg                banded                         794
banded_0005.jpg                banded                         794
banded_0006.jpg                banded                         794
banded_0008.jpg                banded                         905
banded_0009.jpg                banded                         905
banded_0010.jpg                banded                         549
banded_0011.jpg                banded                         794
banded_0012.jpg                banded                         753
banded_0013.jpg                banded                         794
banded_0016.jpg                banded                         794
banded_0019.jpg                banded                         794
banded_0021.jpg                banded   

## 7. Run SHViT's official eval script

DTD's directory tree is already an ImageFolder-style layout — each
texture category lives in its own folder under `dtd/images/<category>/`.
We build a temporary per-class symlink tree from the CoOp split JSON so
that SHViT's `--data-set IMNET` sees the standard `train/<class>/...`
arrangement. Expect ~zero accuracy relative to ImageNet-1K classes —
fine-tuning happens in Stage 3.


In [10]:
import re, json, os, shutil

with open('/content/SHViT/main.py', 'r') as f:
    content = f.read()
content = re.sub(r"torch\.load\(([^,]+),\s*map_location='cpu'\)",
                 r"torch.load(\1, map_location='cpu', weights_only=False)",
                 content)
with open('/content/SHViT/main.py', 'w') as f:
    f.write(content)

# Build a per-class symlink tree from the CoOp split JSON so SHViT's
# IMNET data-set finds the train/<class>/<image>.jpg layout it expects.
IM_ROOT = '/content/imnet_dtd_sanity'
SPLIT_JSON = f'{DATA_ROOT}/dtd/split_zhou_DescribableTextures.json'
IMG_ROOT   = f'{DATA_ROOT}/dtd/images'

with open(SPLIT_JSON) as f:
    split = json.load(f)

if os.path.isdir(IM_ROOT):
    shutil.rmtree(IM_ROOT)
for tgt in ('train', 'val'):
    for rel_path, label, classname in split[tgt]:
        cls_safe = classname.replace(' ', '_').replace('/', '_')
        cls_dir = f'{IM_ROOT}/{tgt}/{cls_safe}'
        os.makedirs(cls_dir, exist_ok=True)
        src = f'{IMG_ROOT}/{rel_path}'
        dst = f'{cls_dir}/{os.path.basename(rel_path)}'
        if not os.path.exists(dst):
            os.symlink(src, dst)

!python /content/SHViT/main.py \
    --model shvit_s4 \
    --eval \
    --resume {WEIGHTS_PATH} \
    --data-path {IM_ROOT} \
    --data-set IMNET \
    --batch-size 64 \
    --num_workers 2 \
    --device cuda


Not using distributed mode
Creating model: shvit_s4
number of params: 16588484
/usr/local/lib/python3.12/dist-packages/timm/utils/cuda.py:40: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self._scaler = torch.cuda.amp.GradScaler()
Loading local checkpoint at /content/weights/shvit_s4.pth
<All keys matched successfully>
Evaluating model: shvit_s4
/content/SHViT/engine.py:91: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Test:  [ 0/12]  eta: 0:01:38  loss: 7.9239 (7.9239)  acc1: 0.0000 (0.0000)  acc5: 1.0417 (1.0417)  time: 8.2109  data: 0.3275  max mem: 479
Test:  [10/12]  eta: 0:00:01  loss: 8.2572 (8.2037)  acc1: 0.0000 (0.0000)  acc5: 0.0000 (0.0947)  time: 0.8281  data: 0.1040  max mem: 479
Test:  [11/12]  eta: 0:00:01  loss: 8.1483 (8.1938)  acc1: 0.0000 (0.0000)  acc5: 0.0000 (0.0887)  time

## Next steps — fine-tuning on DTD

To actually train SHViT on DTD, head over to Stage 3 and run
`finetune_shvit_dtd.py`, which uses the Tip-Adapter split JSON,
CLIP normalization, RandAugment / RandomErasing / Mixup / CutMix /
label-smoothing, AGC-style gradient clipping, and cosine LR with warmup
(the same recipe as the SHViT paper, but with `--nb_classes 47`).
